#ALINEAMIENTO MULTIPLE CLUSTAL OMEGA

1. Extraer Accession IDs desde el archivo .txt de BLAST

In [ ]:
import re
from Bio import Entrez
import time
import os

# ---------- CONFIGURACIÓN ----------
archivo_blast = r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\data\blastn_virus_bacteria.txt"  # Cambia esto por tu archivo
top_hits = 5
Entrez.email = "fgarciao2206@gmail.com"  # Reemplaza por tu correo real

# Ruta personalizada para guardar las secuencias FASTA
directorio_salida = r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_3\top_sequences_blastn"
os.makedirs(directorio_salida, exist_ok=True)

# ---------- EXTRAER ACCESSION IDs ----------
def extraer_accessions(archivo_blast, top_hits=5):
    with open(archivo_blast, "r", encoding="utf-8") as file:
        contenido = file.read()

    queries = contenido.split("Query #")
    accesiones_por_query = []

    for query in queries[1:]:  # Omitimos el encabezado
        matches = re.findall(r'\b([A-Z]{1,2}_?\d+\.\d+)\b', query)
        accesiones = list(dict.fromkeys(matches))[:top_hits]  # Únicos y top N
        accesiones_por_query.append(accesiones)

    return accesiones_por_query

# ---------- DESCARGAR SECUENCIAS ----------
def descargar_secuencias(accession_list, nombre_archivo):
    with open(nombre_archivo, "w") as output_handle:
        for acc in accession_list:
            try:
                handle = Entrez.efetch(db="nucleotide", id=acc, rettype="fasta", retmode="text")
                seq_record = handle.read()
                output_handle.write(seq_record)
                time.sleep(0.5)  # Espera para no saturar NCBI
            except Exception as e:
                print(f"Error al descargar {acc}: {e}")

# ---------- EJECUCIÓN ----------
ids_por_query = extraer_accessions(archivo_blast, top_hits)

for i, accesiones in enumerate(ids_por_query, start=1):
    archivo_salida = os.path.join(directorio_salida, f"query_{i}_top5.fasta")
    print(f"Descargando secuencias para Query #{i}...")
    descargar_secuencias(accesiones, archivo_salida)

print("\n✅ ¡Todos los archivos FASTA han sido generados en la carpeta especificada!")


2. Extracción de segmentos en formato FASTA - Genomas completos

In [5]:
import os
from Bio import Entrez, SeqIO

# Configuración
Entrez.email = "tu_email@institucion.edu"  # ¡Obligatorio! Usa tu email real
directorio_salida = r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_3\top_sequences_blastn"
os.makedirs(directorio_salida, exist_ok=True)

# Diccionario con las regiones a extraer (mismo que antes)
regiones_por_query = {
    "Query2_Ebola": [
        ("MT778476.1", 632, 682), ("MT778476.1", 648, 682),
        ("MT778473.1", 634, 684), ("MT778473.1", 650, 684),
        ("MW797157.1", 591, 641), ("MW797157.1", 607, 641),
        ("OR084906.1", 287, 337), ("OR084906.1", 303, 337),
        ("MT778550.1", 631, 681), ("MT778550.1", 647, 681)
    ],
    "Query6_Ebola": [
        ("KY785949.1", 4368, 4717), ("KY785949.1", 4219, 4372), ("KY785949.1", 12619, 12688),
        ("KY785969.1", 4368, 4717), ("KY785969.1", 4219, 4372), ("KY785969.1", 12619, 12688),
        ("KY786027.1", 4368, 4717), ("KY786027.1", 4219, 4372), ("KY786027.1", 12619, 12688)
    ],
    "Query9_Ebola": [
        ("KY785949.1", 4425, 4641), ("KY785949.1", 4217, 4424), ("KY785949.1", 4642, 4717),
        ("KY785969.1", 4425, 4641), ("KY785969.1", 4217, 4424), ("KY785969.1", 4642, 4717),
        ("KY786027.1", 4425, 4641), ("KY786027.1", 4217, 4424), ("KY786027.1", 4642, 4717),
        ("KY471125.1", 4425, 4641), ("KY471125.1", 4217, 4424), ("KY471125.1", 4642, 4717),
        ("KY785948.1", 4425, 4641), ("KY785948.1", 4217, 4424), ("KY785948.1", 4642, 4717)
    ],
    "Query10_Salmonella": [
        ("CP085798.1", 3486284, 3486861),
        ("CP173647.1", 2818, 3395), ("CP173647.1", 2815, 3395),
        ("CP183508.1", 3916630, 3917207),
        ("CP151082.1", 4091375, 4091952),
        ("CP149175.1", 4053968, 4054545)
    ],
    "Query11_Salmonella": [
        ("CP090529.1", 4081977, 4082273), ("CP090529.1", 4082327, 4082479),
        ("CP090529.1", 4081843, 4081980), ("CP090529.1", 4082273, 4082327),
        ("CP065082.1", 4065023, 4065319), ("CP065082.1", 4065373, 4065525),
        ("CP065082.1", 4064889, 4065026), ("CP065082.1", 4065319, 4065373),
        ("CP075116.1", 3887748, 3888044), ("CP075116.1", 3888098, 3888250),
        ("CP075116.1", 3887614, 3887751), ("CP075116.1", 3888044, 3888098),
        ("CP082464.1", 3867925, 3868221), ("CP082464.1", 3868275, 3868427),
        ("CP082464.1", 3867792, 3867928), ("CP082464.1", 3868221, 3868275),
        ("CP075106.1", 4025529, 4025825), ("CP075106.1", 4025879, 4026031),
        ("CP075106.1", 4025395, 4025532), ("CP075106.1", 4025825, 4025879)
    ]
}


def extraer_y_formatear_regiones(regiones, nombre_archivo):
    """Extrae regiones desde NCBI y guarda con formato personalizado"""
    with open(nombre_archivo, "w") as handle:
        for acc, start, end in regiones:
            try:
                # Descargar la secuencia completa para obtener la descripción
                handle_seq = Entrez.efetch(
                    db="nucleotide",
                    id=acc,
                    rettype="gb",
                    retmode="text"
                )
                record = SeqIO.read(handle_seq, "gb")
                descripcion = record.description
                
                # Descargar solo la región de interés
                handle_region = Entrez.efetch(
                    db="nucleotide",
                    id=acc,
                    rettype="fasta",
                    strand=1,
                    seq_start=start,
                    seq_stop=end
                )
                secuencia = handle_region.read().split("\n", 1)[1].replace("\n", "")
                
                # Escribir en el formato deseado
                handle.write(f">{acc}:{start}-{end} {descripcion}\n{secuencia}\n")
                print(f"✅ Descargado y formateado: {acc} ({start}-{end})")
                
            except Exception as e:
                print(f"❌ Error con {acc}: {e}")

# Procesamiento principal
print("🔍 Iniciando descarga y formateo de secuencias...")
for query, regiones in regiones_por_query.items():
    archivo_fasta = os.path.join(directorio_salida, f"{query}_formateado.fasta")
    print(f"\n📥 Procesando {query} ({len(regiones)} regiones)...")
    extraer_y_formatear_regiones(regiones, archivo_fasta)

print(f"\n🎯 ¡Proceso completado! Archivos en:\n{directorio_salida}")

🔍 Iniciando descarga y formateo de secuencias...

📥 Procesando Query2_Ebola (10 regiones)...
✅ Descargado y formateado: MT778476.1 (632-682)
✅ Descargado y formateado: MT778476.1 (648-682)
✅ Descargado y formateado: MT778473.1 (634-684)
✅ Descargado y formateado: MT778473.1 (650-684)
✅ Descargado y formateado: MW797157.1 (591-641)
✅ Descargado y formateado: MW797157.1 (607-641)
✅ Descargado y formateado: OR084906.1 (287-337)
✅ Descargado y formateado: OR084906.1 (303-337)
✅ Descargado y formateado: MT778550.1 (631-681)
✅ Descargado y formateado: MT778550.1 (647-681)

📥 Procesando Query6_Ebola (9 regiones)...
✅ Descargado y formateado: KY785949.1 (4368-4717)
✅ Descargado y formateado: KY785949.1 (4219-4372)
✅ Descargado y formateado: KY785949.1 (12619-12688)
✅ Descargado y formateado: KY785969.1 (4368-4717)
✅ Descargado y formateado: KY785969.1 (4219-4372)
✅ Descargado y formateado: KY785969.1 (12619-12688)
✅ Descargado y formateado: KY786027.1 (4368-4717)
✅ Descargado y formateado: KY7

3. Multinalineamiento con Clustal Omega

In [6]:
import os
import subprocess

# Carpeta con los .fasta descargados desde BLAST
directorio_fasta = r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_3\top_sequences_blastn"
# Carpeta para guardar los alineamientos
directorio_salida = os.path.join(directorio_fasta, "alineamientos")
os.makedirs(directorio_salida, exist_ok=True)

# Iterar sobre todos los .fasta y ejecutar Clustal Omega
for archivo in os.listdir(directorio_fasta):
    if archivo.endswith(".fasta"):
        archivo_fasta = os.path.join(directorio_fasta, archivo)
        archivo_aln = os.path.join(directorio_salida, archivo.replace(".fasta", ".aln"))

        print(f"Alineando {archivo}...")

        comando = [
            "clustalo",            # Está en PATH
            "-i", archivo_fasta,
            "-o", archivo_aln,
            "--outfmt=clu",        # Formato CLUSTAL
            "--force",             # Sobreescribir si existe
            "--wrap=80"            # Líneas de 80 caracteres
        ]

        try:
            subprocess.run(comando, check=True)
        except subprocess.CalledProcessError as e:
            print(f"❌ Error al alinear {archivo}: {e}")

print("\n✅ ¡Todos los alineamientos fueron generados exitosamente!")



Alineando Query10_Salmonella_formateado.fasta...
Alineando Query11_Salmonella_formateado.fasta...
Alineando Query2_Ebola_formateado.fasta...
Alineando Query6_Ebola_formateado.fasta...
Alineando Query9_Ebola_formateado.fasta...
Alineando query_12_top5.fasta...
❌ Error al alinear query_12_top5.fasta: Command '['clustalo', '-i', 'C:\\Users\\fgarc\\OneDrive\\Escritorio\\Doctorado\\Ramos\\1° Semestre\\Troncal\\proyecto_troncal2\\results\\2025-03-30_FG\\identificacion_patogeno\\identificacion_patogeno_3\\top_sequences_blastn\\query_12_top5.fasta', '-o', 'C:\\Users\\fgarc\\OneDrive\\Escritorio\\Doctorado\\Ramos\\1° Semestre\\Troncal\\proyecto_troncal2\\results\\2025-03-30_FG\\identificacion_patogeno\\identificacion_patogeno_3\\top_sequences_blastn\\alineamientos\\query_12_top5.aln', '--outfmt=clu', '--force', '--wrap=80']' returned non-zero exit status 1.
Alineando query_13_top5.fasta...
❌ Error al alinear query_13_top5.fasta: Command '['clustalo', '-i', 'C:\\Users\\fgarc\\OneDrive\\Escritori